# Task 4 — Neo4j sink

The saved Cypher results include the initial full load and the final state after
the controlled source change in Task 6.


In [1]:
from pathlib import Path
import json
from IPython.display import display

def find_evidence():
    start = Path.cwd().resolve()
    for root in (start, *start.parents):
        candidate = root / 'docs' / 'evidence' / 'live_pipeline_summary.json'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('docs/evidence/live_pipeline_summary.json not found')

evidence_path = find_evidence()
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
task = evidence['task4']
display({
    'captured_at': evidence['captured_at'],
    'event_time_guard': task['event_time_guard'],
    'initial_full_load': task['initial_full_load'],
    'final_post_modification': task['final_post_modification'],
    'target_file': task['target_file'],
})

{'captured_at': '2026-07-25T03:30:43.726347Z',
 'event_time_guard': {'node_edge_acceptance': 'first_load_or_same_hash_or_strictly_newer_event_time',
  'metadata_acceptance': 'non_stale_event_time',
  'late_old_revision_blocked': True,
  'same_revision_replay_allowed': True,
  'neo4j_explain_queries_passed': 3},
 'initial_full_load': {'source_files': 490,
  'nodes': 655365,
  'distinct_node_ids': 655365,
  'edges': 830472,
  'distinct_edge_ids': 830472,
  'duplicate_node_groups': 0,
  'duplicate_edge_groups': 0,
  'unresolved_internal_placeholders': 0},
 'final_post_modification': {'source_files': 490,
  'nodes': 655388,
  'distinct_node_ids': 655388,
  'edges': 830500,
  'distinct_edge_ids': 830500,
  'duplicate_node_groups': 0,
  'duplicate_edge_groups': 0},
 'target_file': {'file_path': 'src/lerobot/__init__.py',
  'file_id': 'file_4bcb0fb8af5f208dad26ab6d584c0d2a8e7c995ea7954b70e1d1976ce286f236',
  'baseline_nodes': 58,
  'baseline_edges': 60,
  'modified_nodes': 81,
  'modified_edg

In [2]:
initial = task['initial_full_load']
final = task['final_post_modification']
target = task['target_file']
guard = task['event_time_guard']
for state in (initial, final):
    assert state['nodes'] == state['distinct_node_ids']
    assert state['edges'] == state['distinct_edge_ids']
    assert state['duplicate_node_groups'] == 0
    assert state['duplicate_edge_groups'] == 0
assert initial['unresolved_internal_placeholders'] == 0
assert initial['source_files'] == final['source_files'] == 490
assert final['nodes'] - initial['nodes'] == target['modified_nodes'] - target['baseline_nodes'] == 23
assert final['edges'] - initial['edges'] == target['modified_edges'] - target['baseline_edges'] == 28
assert target['old_hash_absent_after_reconciliation'] is True
assert guard['late_old_revision_blocked'] is True
assert guard['same_revision_replay_allowed'] is True
assert guard['neo4j_explain_queries_passed'] == 3
display({
    'status': 'PASS',
    'initial_nodes_edges': (initial['nodes'], initial['edges']),
    'final_nodes_edges': (final['nodes'], final['edges']),
    'controlled_delta': {'nodes': 23, 'edges': 28},
    'duplicate_groups': {'nodes': 0, 'edges': 0},
    'late_old_revision_blocked': True,
})

{'status': 'PASS',
 'initial_nodes_edges': (655365, 830472),
 'final_nodes_edges': (655388, 830500),
 'controlled_delta': {'nodes': 23, 'edges': 28},
 'duplicate_groups': {'nodes': 0, 'edges': 0},
 'late_old_revision_blocked': True}

## Reflection

Uniqueness constraints and `MERGE` made exact delivery retries safe. Successful metadata supplied the revision boundary needed to remove old-hash elements after a source edit, while failed parses preserve the last valid graph.